# Cybersecurity Feature Selection, PCR, and PLSR Modeling

**Author:** Usha Priya Krishnasamy  
**Project Type:** Feature selection, dimensionality reduction, and regression-based baseline modeling  
**Datasets:** BETH Honeypot, Cybersecurity Attacks Dataset, and UNSW-NB15

#### Project Overview

This notebook compares feature selection and dimensionality-reduction modeling methods across three cybersecurity datasets used for threat detection, anomaly analysis, and attack classification.

The workflow includes forward selection, backward elimination, Principal Component Regression, and Partial Least Squares Regression.

Although the main cybersecurity objective is classification, these regression-based methods are used as baseline modeling and feature-diagnostic tools to understand feature usefulness, redundancy, and component-based representations.

#### Note

This notebook is included in the modeling folder because it focuses on model development, feature selection, dimensionality reduction, and performance comparison.

The raw datasets are not included in this repository. The analysis was developed using public cybersecurity datasets.

The three datasets under use are 
1. BETH dataset – Contains network traffic records labeled for different attack types, focusing on distinguishing benign vs. malicious activities.
2. Cybersecurity Attacks dataset – Provides additional records of real-world attack behaviors, enabling comparison and model generalization across datasets.
3. UNSW-NB15 dataset – A widely used benchmark dataset for network intrusion detection, including features such as packet statistics, byte counts, and flow information, with multiple classes of cyberattacks.

In [1]:
# Dataset loading
## General libraries 
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

### Libraries for train test split and models

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


########  polynomial libraires
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.stats.outliers_influence import variance_inflation_factor

#########  Linraies for Regularised regression
 
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

##  Week 3 modules
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import GridSearchCV
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.model_selection import cross_val_score

## LOAD DATASETS and specific libraries
 
import kagglehub
import os

# DATASET 1 :  BETH cyberseurity dataset
beth_dataset_path = kagglehub.dataset_download("katehighnam/beth-dataset")
print("Path to dataset files:", beth_dataset_path)
print(os.listdir(beth_dataset_path))
kernel_files = [
    "labelled_training_data.csv",
    "labelled_validation_data.csv",
    "labelled_testing_data.csv"
]
dataframes = [pd.read_csv(os.path.join(beth_dataset_path, file)) for file in kernel_files]
df_beth_merged = pd.concat(dataframes, ignore_index=True)


# DATASET 2: Cybersecurity Dataset
CSA_dataset_path = kagglehub.dataset_download("teamincribo/cyber-security-attacks")
print("Path to dataset files:", CSA_dataset_path)
print(os.listdir(CSA_dataset_path))
csa_csv_path = os.path.join(CSA_dataset_path, "cybersecurity_attacks.csv")
df_csa = pd.read_csv(csa_csv_path)


# DATASET 3 : UNSW-NB15 Intrusion Detection Dataset
path = kagglehub.dataset_download("dhoogla/unswnb15")
df_unsw= pd.read_parquet(os.path.join(path, 'UNSW_NB15_training-set.parquet'))
df_test_unsw=pd.read_parquet(os.path.join(path, 'UNSW_NB15_testing-set.parquet'))


c:\Users\phxlab\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\phxlab\.cache\kagglehub\datasets\katehighnam\beth-dataset\versions\3
['labelled_2021may-ip-10-100-1-105-dns.csv', 'labelled_2021may-ip-10-100-1-105.csv', 'labelled_2021may-ip-10-100-1-186-dns.csv', 'labelled_2021may-ip-10-100-1-186.csv', 'labelled_2021may-ip-10-100-1-26-dns.csv', 'labelled_2021may-ip-10-100-1-26.csv', 'labelled_2021may-ip-10-100-1-4-dns.csv', 'labelled_2021may-ip-10-100-1-4.csv', 'labelled_2021may-ip-10-100-1-95-dns.csv', 'labelled_2021may-ip-10-100-1-95.csv', 'labelled_2021may-ubuntu-dns.csv', 'labelled_2021may-ubuntu.csv', 'labelled_testing_data.csv', 'labelled_training_data.csv', 'labelled_validation_data.csv']
Path to dataset files: C:\Users\phxlab\.cache\kagglehub\datasets\teamincribo\cyber-security-attacks\versions\32
['cybersecurity_attacks.csv', 'README.md']


# 1. BETH Cybersecurity dataset

In [2]:

# Encode categorical features to numerical features using label encoder to allow regression modelling
numeric_columns_beth = df_beth_merged.select_dtypes(include=['number']).columns
categorical_columns_beth = df_beth_merged.select_dtypes(include=['object']).columns
le_dict = {}
for col in categorical_columns_beth:
    le_dict[col] = LabelEncoder()
    df_beth_merged[col] = le_dict[col].fit_transform(df_beth_merged[col])
# Verify and split the datset 
df_train_val,df_test = train_test_split(df_beth_merged, test_size=0.2, stratify=df_beth_merged['evil'], random_state=42)
df_beth, df_val = train_test_split(df_train_val, test_size=0.25, stratify=df_train_val['evil'], random_state=42)
#data cleaning
df_beth.head()
print(f"Null values: {df_beth.isnull().sum()}")
print(f"Shape of the DataFrame: {df_beth.shape}")

numeric_columns_beth = df_beth.select_dtypes(include=['number']).columns
print (f"numeric columns: \n{numeric_columns_beth}")

categorical_columns_beth = df_beth.select_dtypes(include=['object']).columns
print(f"categorical columns: \n {categorical_columns_beth}")


#######################  Assign features and target ################################################################################3
# Features and target (using 'evil' as the target variable)Although the primary target (evil) is binary, 
# we focus on applying linear regression techniques on continuous features to understand model concepts like multicollinearity and interaction effects.
##################################################################################################################################
X_train_beth = df_beth.drop(['evil','sus'],axis=1)
y_train_beth = df_beth['evil']
 
X_val_beth = df_val.drop(['evil', 'sus'], axis=1)
y_val_beth = df_val['evil']

X_test_beth = df_test.drop(['evil', 'sus'], axis=1)
y_test_beth = df_test['evil']


###############################################################SCaling features #################################3
scaler = StandardScaler()
X_train_beth_scaled = scaler.fit_transform(X_train_beth)
X_test_beth_scaled = scaler.transform(X_test_beth)

X_train_beth_df = pd.DataFrame(X_train_beth, columns=X_train_beth.columns)
X_train_beth_scaled_df = pd.DataFrame(X_train_beth_scaled, columns=X_train_beth.columns)
X_test_beth_df = pd.DataFrame(X_test_beth, columns=X_test_beth.columns)
X_test_beth_scaled_df = pd.DataFrame(X_test_beth_scaled, columns=X_test_beth.columns)



Null values: timestamp          0
processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
processName        0
hostName           0
eventId            0
eventName          0
stackAddresses     0
argsNum            0
returnValue        0
args               0
sus                0
evil               0
dtype: int64
Shape of the DataFrame: (684646, 16)
numeric columns: 
Index(['timestamp', 'processId', 'threadId', 'parentProcessId', 'userId',
       'mountNamespace', 'processName', 'hostName', 'eventId', 'eventName',
       'stackAddresses', 'argsNum', 'returnValue', 'args', 'sus', 'evil'],
      dtype='object')
categorical columns: 
 Index([], dtype='object')


#### Target Selection Note

The BETH dataset contains binary cybersecurity labels, including `evil` and `sus`.

In this section, `evil` is used as the target variable. Because `evil` is a binary 0/1 label, these regression-based methods should be interpreted as baseline diagnostic models rather than final classification models.

Final attack-detection modeling should use classification models and classification metrics such as recall, F1-score, ROC-AUC, and confusion matrices.

In [3]:


# --- Forward selection on BETH dataset ---
def forward_selection(X, y):
    initial_features = []
    remaining = list(X.columns)
    best_features = []
    
    while remaining:
        aic_with_candidates = []
        for candidate in remaining:
            features = initial_features + [candidate]
            X_const = sm.add_constant(X[features])
            model = sm.OLS(y, X_const).fit()
            aic_with_candidates.append((model.aic, candidate))
        
        aic_with_candidates.sort()
        best_new_feature = aic_with_candidates[0][1]
        initial_features.append(best_new_feature)
        remaining.remove(best_new_feature)
        best_features.append((aic_with_candidates[0][0], list(initial_features)))
    
    return best_features[-1][1]

# --- Run forward selection ---
selected_forward = forward_selection(X_train_beth_df, y_train_beth)
print("Forward Selection Chose:", selected_forward)

# --- Fit final model ---
X_train_beth_fs = sm.add_constant(X_train_beth_df[selected_forward])
X_test_beth_fs = sm.add_constant(X_test_beth_df[selected_forward])
model_beth_fs = sm.OLS(y_train_beth, X_train_beth_fs).fit()
y_pred_beth_fs = model_beth_fs.predict(X_test_beth_fs)

# --- Evaluate ---
print("\nForward Selection Model Performance:")
print(f"R²: {r2_score(y_test_beth, y_pred_beth_fs):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_fs)):.4f}")


Forward Selection Chose: ['userId', 'processName', 'parentProcessId', 'hostName', 'threadId', 'timestamp', 'processId', 'eventId', 'eventName', 'argsNum', 'args', 'stackAddresses', 'returnValue', 'mountNamespace']

Forward Selection Model Performance:
R²: 0.9668
RMSE: 0.0630


In [4]:
 

# --- Backward elimination on BETH dataset ---
def backward_elimination(X, y):
    features = list(X.columns)
    while len(features) > 0:
        X_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_const).fit()
        pvalues = model.pvalues.iloc[1:]  # Skip constant
        
        # Remove the feature with the highest p-value if it's > 0.05
        if pvalues.max() > 0.05:
            worst_feature = pvalues.idxmax()
            features.remove(worst_feature)
        else:
            break
    return features

# --- Run backward elimination ---
selected_backward = backward_elimination(X_train_beth_df, y_train_beth)
print("Backward Elimination Chose:", selected_backward)

# --- Fit final model ---
X_train_beth_bs = sm.add_constant(X_train_beth_df[selected_backward])
X_test_beth_bs = sm.add_constant(X_test_beth_df[selected_backward])
model_beth_bs = sm.OLS(y_train_beth, X_train_beth_bs).fit()
y_pred_beth_bs = model_beth_bs.predict(X_test_beth_bs)

# --- Evaluate ---
print("\nBackward Elimination Model Performance:")
print(f"R²: {r2_score(y_test_beth, y_pred_beth_bs):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_bs)):.4f}")


Backward Elimination Chose: ['timestamp', 'processId', 'threadId', 'parentProcessId', 'userId', 'mountNamespace', 'processName', 'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum', 'returnValue', 'args']

Backward Elimination Model Performance:
R²: 0.9668
RMSE: 0.0630


In [ ]:
 
# PCA transformation
pca = PCA()
X_train_beth_pca = pca.fit_transform(X_train_beth_scaled_df)
X_test_beth_pca = pca.transform(X_test_beth_scaled_df)

# Find optimal number of components using cross-validated MSE
mse_list = []
for i in range(1, X_train_beth_pca.shape[1] + 1):
    model = LinearRegression()
    mse = -1 * cross_val_score(model, X_train_beth_pca[:, :i], y_train_beth, scoring='neg_mean_squared_error', cv=5).mean()
    mse_list.append(mse)

optimal_components = np.argmin(mse_list) + 1
print(f"Optimal # of PCR components (BETH): {optimal_components}")

# Fit final PCR model
model_pcr = LinearRegression()
model_pcr.fit(X_train_beth_pca[:, :optimal_components], y_train_beth)
y_pred_beth_pcr = model_pcr.predict(X_test_beth_pca[:, :optimal_components])

# Evaluate PCR
print("\nPCR Model Performance (BETH):")
print(f"R²: {r2_score(y_test_beth, y_pred_beth_pcr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_pcr)):.4f}")
#print( X_test_beth_scaled_df.shape, y_test_beth)


Optimal # of PCR components (BETH): 14

PCR Model Performance (BETH):
R²: 0.9671
RMSE: 0.0628
(228216, 14) 997158    1
240245    0
340744    0
360890    0
281085    0
         ..
724498    0
965389    0
441507    0
842809    0
437701    0
Name: evil, Length: 228216, dtype: int64


In [6]:
 
# Find optimal number of PLS components using cross-validated MSE
mse_pls = []
for i in range(1, X_train_beth_scaled_df.shape[1] + 1):
    pls = PLSRegression(n_components=i)
    mse = -1 * cross_val_score(pls, X_train_beth_scaled_df, y_train_beth, scoring='neg_mean_squared_error', cv=5).mean()
    mse_pls.append(mse)

optimal_pls = np.argmin(mse_pls) + 1
print(f"Optimal # of PLS components (BETH): {optimal_pls}")

# Fit final PLSR model
pls_beth_final = PLSRegression(n_components=optimal_pls)
pls_beth_final.fit(X_train_beth_scaled_df, y_train_beth)
y_pred_beth_pls = pls_beth_final.predict(X_test_beth_scaled_df)

# Evaluate PLSR
print("\nPLSR Model Performance (BETH):")
print(f"R²: {r2_score(y_test_beth, y_pred_beth_pls):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_pls)):.4f}")


Optimal # of PLS components (BETH): 14

PLSR Model Performance (BETH):
R²: 0.9671
RMSE: 0.0628


In [7]:
 

# --- BETH dataset performance summary ---
results_beth = pd.DataFrame({
    'Model': ['Forward Selection', 'Backward Elimination', 'PCR', 'PLSR'],
    'R² Score': [
        r2_score(y_test_beth, y_pred_beth_fs),
        r2_score(y_test_beth, y_pred_beth_bs),
        r2_score(y_test_beth, y_pred_beth_pcr),
        r2_score(y_test_beth, y_pred_beth_pls)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_fs)),
        np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_bs)),
        np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_pcr)),
        np.sqrt(mean_squared_error(y_test_beth, y_pred_beth_pls))
    ]
})

print("\nBETH Dataset Model Comparison:")
display(results_beth)



BETH Dataset Model Comparison:


,Model,R² Score,RMSE
0,Forward Selection,0.966797,0.063008
1,Backward Elimination,0.966797,0.063008
2,PCR,0.967064,0.062754
3,PLSR,0.967064,0.062754


#### BETH Dataset Findings

The BETH dataset showed strong baseline performance across forward selection, backward elimination, PCR, and PLSR.

PCR and PLSR produced slightly stronger results than the feature-selection models, suggesting that component-based methods can capture useful structure in the host-level features.

Because the target `evil` is binary, these results should be interpreted as regularized linear baseline diagnostics rather than final classification results.

The selected and component-based results indicate that process, user, event, host, and namespace-related variables contain useful signal for identifying malicious activity.

# 2.Cybersecurity dataset

In [8]:
# Dataset loading 
print(df_csa.columns.to_list())
cols_to_dropcsa = ['Source IP Address','Destination IP Address','User Information', 'Device Information','Geo-location Data','Proxy Information','Log Source','Payload Data']
df_new_csa = df_csa.drop(columns=cols_to_dropcsa)
df_new_csa["Malware Indicators"] = df_new_csa["Malware Indicators"].fillna("None Detected")
df_new_csa["Alerts/Warnings"] = df_new_csa["Alerts/Warnings"].fillna("No Alert")

df_csa_encoded = df_new_csa.copy()

# Define categorical columns (assuming csa_categorical_columns is defined elsewhere)
csa_categorical_columns = [ 'Protocol', 'Packet Type', 'Traffic Type', 'Malware Indicators', 'Alerts/Warnings', 'Attack Type', 'Attack Signature', 'Action Taken', 'Severity Level', 'Network Segment', 'Firewall Logs', 'IDS/IPS Alerts']

# Convert Timestamp to datetime and verify
df_csa_encoded['Timestamp'] = pd.to_datetime(df_csa_encoded['Timestamp'], errors='coerce')
if df_csa_encoded['Timestamp'].dtype != 'datetime64[ns]':
    print("Warning: Timestamp column is not datetime64[ns]. Checking for issues...")
    print(df_csa_encoded['Timestamp'].head())  # Debug: Check the first few values
    df_csa_encoded = df_csa_encoded.dropna(subset=['Timestamp'])  # Drop rows with NaT if needed

# Extract time components
df_csa_encoded['Hour'] = df_csa_encoded['Timestamp'].dt.hour
df_csa_encoded['Weekday'] = df_csa_encoded['Timestamp'].dt.weekday
df_csa_encoded['Month'] = df_csa_encoded['Timestamp'].dt.month

# Label encode all categorical columns
label_encoders = {}
for col in csa_categorical_columns:
    le = LabelEncoder()
    df_csa_encoded[col] = le.fit_transform(df_csa_encoded[col].astype(str).fillna('Unknown'))
    label_encoders[col] = le

# Combine all numeric and encoded features
df_features = df_csa_encoded.select_dtypes(include='number')

# Handle any remaining NaN values
df_features = df_features.fillna(0)


csa_numeric_columns=df_features.select_dtypes(include='number').columns
print("Numeric columns in the dataset:\n")
print(csa_numeric_columns.to_list())
print('\n')

csa_categorical_columns=df_features.select_dtypes(include='object').columns
print("Categorical columns in the dataset:\n")
print(csa_categorical_columns.to_list())

######################################################################################################


X_train_csa, X_test_csa, y_train_csa, y_test_csa = train_test_split(df_features.drop(columns=['Anomaly Scores']), df_features['Anomaly Scores'], test_size=0.2, random_state=0)


###############################################################SCaling features #################################3
scaler = StandardScaler()
X_train_csa_scaled = scaler.fit_transform(X_train_csa)
X_test_csa_scaled = scaler.transform(X_test_csa)

X_train_csa_df = pd.DataFrame(X_train_csa, columns=X_train_csa.columns)
X_train_csa_scaled_df = pd.DataFrame(X_train_csa_scaled, columns=X_train_csa.columns)
X_test_csa_df = pd.DataFrame(X_test_csa, columns=X_test_csa.columns)
X_test_csa_scaled_df = pd.DataFrame(X_test_csa_scaled, columns=X_test_csa.columns)

['Timestamp', 'Source IP Address', 'Destination IP Address', 'Source Port', 'Destination Port', 'Protocol', 'Packet Length', 'Packet Type', 'Traffic Type', 'Payload Data', 'Malware Indicators', 'Anomaly Scores', 'Alerts/Warnings', 'Attack Type', 'Attack Signature', 'Action Taken', 'Severity Level', 'User Information', 'Device Information', 'Network Segment', 'Geo-location Data', 'Proxy Information', 'Firewall Logs', 'IDS/IPS Alerts', 'Log Source']
Numeric columns in the dataset:

['Source Port', 'Destination Port', 'Protocol', 'Packet Length', 'Packet Type', 'Traffic Type', 'Malware Indicators', 'Anomaly Scores', 'Alerts/Warnings', 'Attack Type', 'Attack Signature', 'Action Taken', 'Severity Level', 'Network Segment', 'Firewall Logs', 'IDS/IPS Alerts', 'Hour', 'Weekday', 'Month']


Categorical columns in the dataset:

[]


#### Target Selection Note

The Cybersecurity Attacks dataset uses `Anomaly Scores` as a continuous target in this section.

This makes the dataset suitable for regression-based feature selection, PCR, and PLSR comparison. However, the weak model performance should be interpreted carefully because anomaly behavior may not follow a simple linear structure.

In [9]:
 

# --- Forward selection on CSA dataset ---
def forward_selection(X, y):
    initial_features = []
    remaining = list(X.columns)
    best_features = []
    
    while remaining:
        aic_with_candidates = []
        for candidate in remaining:
            features = initial_features + [candidate]
            X_const = sm.add_constant(X[features])
            model = sm.OLS(y, X_const).fit()
            aic_with_candidates.append((model.aic, candidate))
        
        aic_with_candidates.sort()
        best_new_feature = aic_with_candidates[0][1]
        initial_features.append(best_new_feature)
        remaining.remove(best_new_feature)
        best_features.append((aic_with_candidates[0][0], list(initial_features)))
    
    return best_features[-1][1]

# --- Run forward selection ---
selected_forward_csa = forward_selection(X_train_csa_df, y_train_csa)
print("Forward Selection Chose (CSA):", selected_forward_csa)

# --- Fit final model ---
X_train_fs_csa = sm.add_constant(X_train_csa_df[selected_forward_csa])
X_test_fs_csa = sm.add_constant(X_test_csa_df[selected_forward_csa])
model_fs_csa = sm.OLS(y_train_csa, X_train_fs_csa).fit()
y_pred_fs_csa = model_fs_csa.predict(X_test_fs_csa)

# --- Evaluate ---
print("\nForward Selection Model Performance (CSA):")
print(f"R²: {r2_score(y_test_csa, y_pred_fs_csa):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_csa, y_pred_fs_csa)):.4f}")



Forward Selection Chose (CSA): ['Packet Type', 'Network Segment', 'Severity Level', 'Destination Port', 'IDS/IPS Alerts', 'Hour', 'Source Port', 'Month', 'Packet Length', 'Action Taken', 'Malware Indicators', 'Attack Signature', 'Protocol', 'Traffic Type', 'Weekday', 'Firewall Logs', 'Attack Type', 'Alerts/Warnings']

Forward Selection Model Performance (CSA):
R²: -0.0007
RMSE: 28.7707


In [10]:

# --- Backward elimination on CSA dataset ---
def backward_elimination(X, y):
    features = list(X.columns)
    while len(features) > 0:
        X_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_const).fit()
        pvalues = model.pvalues.iloc[1:]  # Skip constant
        
        if pvalues.max() > 0.05:
            worst_feature = pvalues.idxmax()
            features.remove(worst_feature)
        else:
            break
    return features

# --- Run backward elimination ---
selected_backward_csa = backward_elimination(X_train_csa_df, y_train_csa)
print("Backward Elimination Chose (CSA):", selected_backward_csa)

# --- Fit final model ---
X_train_bs_csa = sm.add_constant(X_train_csa_df[selected_backward_csa])
X_test_bs_csa = sm.add_constant(X_test_csa_df[selected_backward_csa])
model_bs_csa = sm.OLS(y_train_csa, X_train_bs_csa).fit()
y_pred_bs_csa = model_bs_csa.predict(X_test_bs_csa)

# --- Evaluate ---
print("\nBackward Elimination Model Performance (CSA):")
print(f"R²: {r2_score(y_test_csa, y_pred_bs_csa):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_csa, y_pred_bs_csa)):.4f}")


Backward Elimination Chose (CSA): []

Backward Elimination Model Performance (CSA):
R²: -0.0001
RMSE: 28.7626


In [11]:
 

# PCA transformation
pca = PCA()
X_train_csa_pca = pca.fit_transform(X_train_csa_scaled_df)
X_test_csa_pca = pca.transform(X_test_csa_scaled_df)

# Find optimal number of components using cross-validated MSE
mse_list = []
for i in range(1, X_train_csa_pca.shape[1] + 1):
    model = LinearRegression()
    mse = -1 * cross_val_score(model, X_train_csa_pca[:, :i], y_train_csa, scoring='neg_mean_squared_error', cv=5).mean()
    mse_list.append(mse)

optimal_components = np.argmin(mse_list) + 1
print(f"Optimal # of PCR components (CSA): {optimal_components}")

# Fit final PCR model
model_csa_pcr = LinearRegression()
model_csa_pcr.fit(X_train_csa_pca[:, :optimal_components], y_train_csa)
y_pred_csa_pcr = model_csa_pcr.predict(X_test_csa_pca[:, :optimal_components])

# Evaluate PCR
print("\nPCR Model Performance (CSA):")
print(f"R²: {r2_score(y_test_csa, y_pred_csa_pcr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_csa, y_pred_csa_pcr)):.4f}")


Optimal # of PCR components (CSA): 1

PCR Model Performance (CSA):
R²: -0.0001
RMSE: 28.7623


In [12]:
# Find optimal number of PLS components using cross-validated MSE
mse_pls = []
for i in range(1, X_train_csa_scaled_df.shape[1] + 1):
    pls = PLSRegression(n_components=i)
    mse = -1 * cross_val_score(pls, X_train_csa_scaled_df, y_train_csa, scoring='neg_mean_squared_error', cv=5).mean()
    mse_pls.append(mse)

optimal_pls = np.argmin(mse_pls) + 1
print(f"Optimal # of PLS components (CSA): {optimal_pls}")

# Fit final PLSR model
pls_csa_final = PLSRegression(n_components=optimal_pls)
pls_csa_final.fit(X_train_csa_scaled_df, y_train_csa)
y_pred_csa_pls = pls_csa_final.predict(X_test_csa_scaled_df)

# Evaluate PLSR
print("\nPLSR Model Performance (CSA):")
print(f"R²: {r2_score(y_test_csa, y_pred_csa_pls):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_csa, y_pred_csa_pls)):.4f}")

Optimal # of PLS components (CSA): 1

PLSR Model Performance (CSA):
R²: -0.0007
RMSE: 28.7706


In [13]:
# --- CSA dataset performance summary ---
results_csa = pd.DataFrame({
    'Model': ['Forward Selection', 'Backward Elimination', 'PCR', 'PLSR'],
    'R² Score': [
        r2_score(y_test_csa, y_pred_fs_csa),
        r2_score(y_test_csa, y_pred_bs_csa),
        r2_score(y_test_csa, y_pred_csa_pcr),
        r2_score(y_test_csa, y_pred_csa_pls)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test_csa, y_pred_fs_csa)),
        np.sqrt(mean_squared_error(y_test_csa, y_pred_bs_csa)),
        np.sqrt(mean_squared_error(y_test_csa, y_pred_csa_pcr)),
        np.sqrt(mean_squared_error(y_test_csa, y_pred_csa_pls))
    ]
})

print("\nCSA Dataset Model Comparison:")
display(results_csa)




CSA Dataset Model Comparison:


,Model,R² Score,RMSE
0,Forward Selection,-0.000665,28.770659
1,Backward Elimination,-0.000104,28.762603
2,PCR,-0.000084,28.762315
3,PLSR,-0.000664,28.770643


#### Cybersecurity Attacks Dataset Findings

The Cybersecurity Attacks dataset showed weak performance across forward selection, backward elimination, PCR, and PLSR.

R² values near zero or slightly negative indicate that these regression-based methods do not explain anomaly scores well.

This suggests that the relationship between the available features and anomaly scores is weak, noisy, or non-linear.

The dataset may still be useful for exploratory analysis and pipeline testing, but stronger non-linear models or anomaly detection methods are needed for meaningful threat detection.

# 3.UNSW-NB15 Intrusion Detection dataset

In [14]:
# Dataset loading
unsw_numeric_columns=df_unsw.select_dtypes(include='number').columns

unsw_categorical_columns=df_unsw.select_dtypes(include=['object', 'category']).columns

# Combine training and test sets for preprocessing
df_unsw_combined = pd.concat([df_unsw, df_test_unsw], axis=0, ignore_index=True)

# Encode categorical columns
 
label_encoders = {}
for col in unsw_categorical_columns:
    le = LabelEncoder()
    df_unsw_combined[col] = le.fit_transform(df_unsw_combined[col].astype(str).fillna('Unknown'))
    label_encoders[col] = le

# Features and target (using 'label' as the target variable)
X = df_unsw_combined[unsw_numeric_columns.tolist() + unsw_categorical_columns.tolist()].drop(columns=['label'])
y = df_unsw_combined['label']

#####################################################################Split into train and test sets
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


unsw_numeric=df_unsw_combined.select_dtypes(include='number').columns
print("Numeric columns in the dataset:\n")
print(unsw_numeric.to_list())
print('\n')

unsw_categorical=df_unsw_combined.select_dtypes(include='object').columns
print("Categorical columns in the dataset:\n")
print(unsw_categorical.to_list())####################################################Scaling##########################################################

scaler = StandardScaler()
X_train_unsw_scaled = scaler.fit_transform(X_train)
X_test_unsw_scaled = scaler.transform(X_test)

X_train_unsw_df = pd.DataFrame(X_train, columns=X_train.columns)
X_train_unsw_scaled_df = pd.DataFrame(X_train_unsw_scaled, columns=X_train.columns)
X_test_unsw_df = pd.DataFrame(X_test, columns=X_test.columns)
X_test_unsw_scaled_df = pd.DataFrame(X_test_unsw_scaled, columns=X_test.columns)

Numeric columns in the dataset:

['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'attack_cat', 'label']


Categorical columns in the dataset:

[]


#### Target Selection Note

The UNSW-NB15 dataset uses `label` as the target variable.

Because `label` is a binary 0/1 intrusion-detection label, these regression-based methods are used only as baseline diagnostic models. Final intrusion detection should rely on classification models and classification metrics.

In [15]:
 

# --- Forward selection on UNSW dataset ---
def forward_selection(X, y):
    initial_features = []
    remaining = list(X.columns)
    best_features = []
    
    while remaining:
        aic_with_candidates = []
        for candidate in remaining:
            features = initial_features + [candidate]
            X_const = sm.add_constant(X[features])
            model = sm.OLS(y, X_const).fit()
            aic_with_candidates.append((model.aic, candidate))
        
        aic_with_candidates.sort()
        best_new_feature = aic_with_candidates[0][1]
        initial_features.append(best_new_feature)
        remaining.remove(best_new_feature)
        best_features.append((aic_with_candidates[0][0], list(initial_features)))
    
    return best_features[-1][1]

# --- Run forward selection ---
selected_forward_unsw = forward_selection(X_train_unsw_df, y_train)
print("Forward Selection Chose (UNSW):", selected_forward_unsw)

# --- Fit final model ---
X_train_fs_unsw = sm.add_constant(X_train_unsw_df[selected_forward_unsw])
X_test_fs_unsw = sm.add_constant(X_test_unsw_df[selected_forward_unsw])
model_fs_unsw = sm.OLS(y_train, X_train_fs_unsw).fit()
y_pred_fs_unsw = model_fs_unsw.predict(X_test_fs_unsw)

# --- Evaluate ---
print("\nForward Selection Model Performance (UNSW):")
print(f"R²: {r2_score(y_test, y_pred_fs_unsw):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_fs_unsw)):.4f}")


Forward Selection Chose (UNSW): ['attack_cat', 'ct_dst_sport_ltm', 'state', 'dload', 'is_sm_ips_ports', 'ackdat', 'swin', 'dwin', 'proto', 'service', 'smean', 'dmean', 'ct_flw_http_mthd', 'sbytes', 'spkts', 'dbytes', 'dloss', 'sload', 'rate', 'ct_src_dport_ltm', 'sloss', 'djit', 'dpkts', 'dinpkt', 'sinpkt', 'trans_depth', 'response_body_len', 'is_ftp_login', 'ct_ftp_cmd', 'synack', 'stcpb', 'tcprtt', 'dtcpb', 'sjit', 'dur']

Forward Selection Model Performance (UNSW):
R²: 0.6588
RMSE: 0.2807


In [16]:
# --- Backward elimination on UNSW dataset ---
def backward_elimination(X, y):
    features = list(X.columns)
    while len(features) > 0:
        X_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_const).fit()
        pvalues = model.pvalues.iloc[1:]  # Skip constant
        
        if pvalues.max() > 0.05:
            worst_feature = pvalues.idxmax()
            features.remove(worst_feature)
        else:
            break
    return features

# --- Run backward elimination ---
selected_backward_unsw = backward_elimination(X_train_unsw_df, y_train)
print("Backward Elimination Chose (UNSW):", selected_backward_unsw)

# --- Fit final model ---
X_train_bs_unsw = sm.add_constant(X_train_unsw_df[selected_backward_unsw])
X_test_bs_unsw = sm.add_constant(X_test_unsw_df[selected_backward_unsw])
model_bs_unsw = sm.OLS(y_train, X_train_bs_unsw).fit()
y_pred_bs_unsw = model_bs_unsw.predict(X_test_bs_unsw)

# --- Evaluate ---
print("\nBackward Elimination Model Performance (UNSW):")
print(f"R²: {r2_score(y_test, y_pred_bs_unsw):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_bs_unsw)):.4f}")


Backward Elimination Chose (UNSW): ['spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'djit', 'swin', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'proto', 'service', 'state', 'attack_cat']

Backward Elimination Model Performance (UNSW):
R²: 0.6588
RMSE: 0.2807


In [17]:

pca = PCA()
X_train_unsw_pca = pca.fit_transform(X_train_unsw_scaled_df)
X_test_unsw_pca = pca.transform(X_test_unsw_scaled_df)

# --- Find optimal number of components using cross-validated MSE ---
mse_list = []
for i in range(1, X_train_unsw_scaled_df.shape[1] + 1):
    model = LinearRegression()
    mse = -1 * cross_val_score(model, X_train_unsw_pca[:, :i], y_train, scoring='neg_mean_squared_error', cv=5).mean()
    mse_list.append(mse)

optimal_components = np.argmin(mse_list) + 1
print(f"Optimal # of components (PCR) for UNSW: {optimal_components}")

# --- Fit PCR model ---
model_unsw_pcr = LinearRegression()
model_unsw_pcr.fit(X_train_unsw_pca[:, :optimal_components], y_train)
y_pred_unsw_pcr = model_unsw_pcr.predict(X_test_unsw_pca[:, :optimal_components])

# --- Evaluate ---
print("\nPCR Model Performance (UNSW):")
print(f"R²: {r2_score(y_test, y_pred_unsw_pcr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_unsw_pcr)):.4f}")


Optimal # of components (PCR) for UNSW: 35

PCR Model Performance (UNSW):
R²: 0.6588
RMSE: 0.2807


In [18]:
 
# --- Find optimal number of components using cross-validated MSE ---
mse_pls = []
for i in range(1, X_train_unsw_scaled_df.shape[1] + 1):
    pls = PLSRegression(n_components=i)
    mse = -1 * cross_val_score(pls, X_train_unsw_scaled_df, y_train, scoring='neg_mean_squared_error', cv=5).mean()
    mse_pls.append(mse)

optimal_pls = np.argmin(mse_pls) + 1
print(f"Optimal # of components (PLSR) for UNSW: {optimal_pls}")

# --- Fit final PLS model ---
pls_unsw_final = PLSRegression(n_components=optimal_pls)
pls_unsw_final.fit(X_train_unsw_scaled_df, y_train)
y_pred_unsw_pls = pls_unsw_final.predict(X_test_unsw_scaled_df)

# --- Evaluate ---
print("\nPLSR Model Performance (UNSW):")
print(f"R²: {r2_score(y_test, y_pred_unsw_pls):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_unsw_pls)):.4f}")


Optimal # of components (PLSR) for UNSW: 35

PLSR Model Performance (UNSW):
R²: 0.6588
RMSE: 0.2807


In [19]:

# --- UNSW dataset performance summary ---
results_unsw = pd.DataFrame({
    'Model': ['Forward Selection', 'Backward Elimination', 'PCR', 'PLSR'],
    'R² Score': [
        r2_score(y_test, y_pred_fs_unsw),
        r2_score(y_test, y_pred_bs_unsw),
        r2_score(y_test, y_pred_unsw_pcr),
        r2_score(y_test, y_pred_unsw_pls)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_fs_unsw)),
        np.sqrt(mean_squared_error(y_test, y_pred_bs_unsw)),
        np.sqrt(mean_squared_error(y_test, y_pred_unsw_pcr)),
        np.sqrt(mean_squared_error(y_test, y_pred_unsw_pls))
    ]
})

print("\nUNSW Dataset Model Comparison:")
display(results_unsw)



UNSW Dataset Model Comparison:


,Model,R² Score,RMSE
0,Forward Selection,0.658782,0.280716
1,Backward Elimination,0.658780,0.280717
2,PCR,0.658770,0.280721
3,PLSR,0.658770,0.280721


#### UNSW-NB15 Dataset Findings

The UNSW-NB15 dataset produced consistent results across forward selection, backward elimination, PCR, and PLSR.

The similar RMSE and R² values suggest that many network-flow features contain overlapping information. This is expected because packet counts, byte counts, loss counts, and TCP-related features are often highly correlated.

Because the target `label` is binary, these results should be treated as baseline diagnostic results. Final intrusion detection should use classification models such as logistic regression, random forest, gradient boosting, or SVM.


#### Overall Feature Selection, PCR, and PLSR Conclusion

Across the three cybersecurity datasets, feature selection, PCR, and PLSR provided useful baseline information about feature relationships and dimensionality reduction.

BETH showed strong baseline performance, suggesting that host-level process, user, host, event, and namespace features contain useful signal for malicious activity detection.

The Cybersecurity Attacks dataset showed weak regression performance, indicating that anomaly scores are not well explained by simple linear or component-based methods.

UNSW-NB15 showed moderate and consistent performance across all methods, suggesting that many network-flow features are informative but also redundant.

Overall, this notebook supports the need for careful feature selection and dimensionality review before classification modeling. Since BETH and UNSW use binary cybersecurity labels, final model evaluation should use classification algorithms and metrics such as recall, F1-score, ROC-AUC, and confusion matrices.